# Create Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.dlt_capital;

# check after create pipline ans schema

In [0]:
%sql
SELECT (SELECT COUNT(*) FROM workspace.silver.projects)        AS silver_rows,
       (SELECT COUNT(*) FROM workspace.dlt_capital.project_kpis) AS dlt_rows,
       (SELECT COUNT(*) FROM workspace.dlt_capital.projects_quarantine) AS quarantined;

# check quality data

In [0]:
%sql
SELECT row_expectations.dataset,
       row_expectations.name,
       SUM(row_expectations.passed_records) AS passed,
       SUM(row_expectations.failed_records) AS failed
FROM (
  SELECT explode(from_json(
           details:flow_progress.data_quality.expectations,
           "array<struct<name:string,dataset:string,passed_records:int,failed_records:int>>")) AS row_expectations
  FROM event_log(TABLE(workspace.dlt_capital.projects_validated))
  WHERE event_type = 'flow_progress'
)
WHERE row_expectations.name IS NOT NULL
GROUP BY 1, 2
ORDER BY failed DESC;